# Dataset Final v2 — Features complètes
- Komoot surfaces (0 pour colonnes vides si URL présente, NaN si pas d'URL)
- Features GPX : distance, dénivelé, cols, gradient fin de course
- Nouvelles features : profil début de course (50 premiers km)
- Liste des courses sans données Komoot

In [1]:
import os
import re
import numpy as np
import pandas as pd
import gpxpy
from scipy.signal import find_peaks
from tqdm import tqdm

BASE       = '/Users/arthurdeletang/Desktop/Stage M1/Code/data'
GPX_DIR    = os.path.join(BASE, 'gpx_files_2')
MATCH_DIR  = os.path.join(BASE, 'matching', 'all')
KOMOOT_CSV = os.path.join(MATCH_DIR, 'komoot_surface_all.csv')
OUTPUT_CSV = os.path.join(MATCH_DIR, 'dataset_final_v2.csv')

/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ── 1. Charger tous les fichiers matching ─────────────────────────────────────
dfs = []
for year in range(2017, 2026):
    path = os.path.join(MATCH_DIR, f'matching_{year}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df[df['statut'] == 'match']
        dfs.append(df)
        print(f'{year} : {len(df)} courses')

df_match = pd.concat(dfs, ignore_index=True)
df_match = df_match.drop_duplicates(subset='fichier_gpx')
print(f'Total unique : {len(df_match)} courses')

2017 : 595 courses
2018 : 1040 courses
2019 : 1062 courses
2020 : 448 courses
2021 : 700 courses
2022 : 813 courses
2023 : 937 courses
2024 : 890 courses
2025 : 906 courses
Total unique : 7391 courses


In [3]:
# ── 2. Joindre avec Komoot ────────────────────────────────────────────────────
df_komoot = pd.read_csv(KOMOOT_CSV)
if '_km' in df_komoot.columns:
    df_komoot = df_komoot.drop(columns=['_km'])

cols_komoot = [c for c in df_komoot.columns if c not in ['fichier_gpx', 'route_url']]
print(f'Komoot : {len(df_komoot)} lignes, {len(cols_komoot)} colonnes surface')

df = df_match.merge(df_komoot, on='fichier_gpx', how='left')

# Pour les courses AVEC URL Komoot : remplacer NaN des colonnes surface par 0
# Pour les courses SANS URL Komoot : laisser NaN
mask_avec_url = df['route_url'].notna()
df.loc[mask_avec_url, cols_komoot] = df.loc[mask_avec_url, cols_komoot].fillna(0)

print(f'Avec URL Komoot : {mask_avec_url.sum()}')
print(f'Sans URL Komoot : {(~mask_avec_url).sum()}')

Komoot : 7343 lignes, 17 colonnes surface
Avec URL Komoot : 7217
Sans URL Komoot : 174


In [4]:
# ── 3. Courses sans données Komoot ────────────────────────────────────────────
sans_komoot = df[df['route_url'].isna()][['fichier_gpx', 'annee', 'categorie', 'nom_pcs']]
print(f'Courses sans Komoot : {len(sans_komoot)}')
print(sans_komoot['annee'].value_counts().sort_index())
print('\nListe complète :')
pd.set_option('display.max_rows', None)
print(sans_komoot.to_string(index=False))

Courses sans Komoot : 174
annee
2017    12
2018    28
2019    21
2020     5
2021    18
2022    23
2023    23
2024    25
2025    19
Name: count, dtype: int64

Liste complète :
                                                             fichier_gpx  annee categorie                                            nom_pcs
                   2017 Dwars Door Vlaanderen - A travers la Flandre.gpx   2017     1.UWT                              dwars-door-vlaanderen
                                       2017 Tour of Thailand Stage 2.gpx   2017       2.1                                   tour-of-thailand
       2017 GP de Denain - Porte du Hainaut - Valenciennes Metropole.gpx   2017      1.HC                                       gp-de-denain
  2017 4 Jours de Dunkerque - Grand Prix des Hauts-de-France Stage 1.gpx   2017      2.HC                               4-jours-de-dunkerque
  2017 4 Jours de Dunkerque - Grand Prix des Hauts-de-France Stage 2.gpx   2017      2.HC                               

In [5]:
# ── 4. Fonctions extraction GPX ───────────────────────────────────────────────

def parse_gpx_safe(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    content = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#)', '&amp;', content)
    return gpxpy.parse(content)

def find_gpx_path(fname, gpx_dir):
    full_path = os.path.join(gpx_dir, fname)
    if os.path.exists(full_path):
        return full_path
    replaced = fname.replace(' / ', ' - ')
    if os.path.exists(os.path.join(gpx_dir, replaced)):
        return os.path.join(gpx_dir, replaced)
    base = fname.split(' / ')[0].strip()
    for candidate in [base + ' .gpx', base + '.gpx']:
        path = os.path.join(gpx_dir, candidate)
        if os.path.exists(path):
            return path
    return None

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def get_number_peaks(series, prominence):
    store = find_peaks(series, prominence=prominence)
    try:
        if (len(store[0]) > 1) and (np.min(np.diff(store[0])) < prominence):
            return len(store[1]['prominences'][np.diff(np.insert(store[0], 0, 0)) > prominence])
        else:
            return len(store[1]['prominences'])
    except:
        return len(store[1]['prominences'])

def gradient_last_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1, 0, -1):
        dist_cumul += haversine(lats[i-1], lons[i-1], lats[i], lons[i])
        if dist_cumul >= km:
            deniv = elevations[-1] - elevations[i-1]
            return round(deniv / (km * 1000) * 100, 2)
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def gradient_first_km(lats, lons, elevations, km):
    """Gradient moyen sur les X premiers km."""
    dist_cumul = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        if dist_cumul >= km:
            deniv = elevations[i+1] - elevations[0]
            return round(deniv / (km * 1000) * 100, 2)
    # Trace plus courte que X km
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def denivele_first_km(lats, lons, elevations, km):
    """Denivele positif sur les X premiers km."""
    dist_cumul = 0.0
    deniv = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        diff = elevations[i+1] - elevations[i]
        if diff > 0:
            deniv += diff
        if dist_cumul >= km:
            break
    return round(deniv, 0)

def extract_gpx_features(filepath):
    try:
        gpx = parse_gpx_safe(filepath)
        points = gpx.tracks[0].segments[0].points
        elevations = [p.elevation or 0 for p in points]
        lats = [p.latitude for p in points]
        lons = [p.longitude for p in points]

        dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
        elev_diff = np.diff(elevations)
        denivele_pos = float(np.sum(elev_diff[elev_diff > 0]))
        denivele_neg = float(np.abs(np.sum(elev_diff[elev_diff < 0])))
        max_elev = float(np.max(elevations))
        min_elev = float(np.min(elevations))

        elev_series = elevations + [0]
        n = len(elev_series)

        def loc_last(prominence):
            peaks = find_peaks(elev_series, prominence=prominence)[0]
            return float(np.max(peaks) / n) if len(peaks) > 0 else 0.0

        last_points = min(500, len(elevations))
        deniv_last_5km = float(np.sum(d for d in np.diff(elevations[-last_points:]) if d > 0))

        return {
            'distance_gpx_km':      round(dist, 2),
            'denivele_pos':         round(denivele_pos, 0),
            'denivele_neg':         round(denivele_neg, 0),
            'altitude_max':         round(max_elev, 0),
            'altitude_min':         round(min_elev, 0),
            'n_cols_hc':  get_number_peaks(elev_series, 800),
            'n_cols_cat1': get_number_peaks(elev_series, 640) - get_number_peaks(elev_series, 800),
            'n_cols_cat2': get_number_peaks(elev_series, 320) - get_number_peaks(elev_series, 640),
            'n_cols_cat3': get_number_peaks(elev_series, 160) - get_number_peaks(elev_series, 320),
            'n_cols_cat4': get_number_peaks(elev_series, 80)  - get_number_peaks(elev_series, 160),
            'loc_last_col_cat4':    round(loc_last(80), 4),
            'loc_last_col_cat3':    round(loc_last(160), 4),
            'loc_last_col_cat2':    round(loc_last(320), 4),
            'loc_last_col_cat1':    round(loc_last(640), 4),
            'loc_last_col_hc':      round(loc_last(800), 4),
            'gradient_last_1km':    gradient_last_km(lats, lons, elevations, 1),
            'gradient_last_3km':    gradient_last_km(lats, lons, elevations, 3),
            'gradient_last_5km':    gradient_last_km(lats, lons, elevations, 5),
            'denivele_last_5km':    round(deniv_last_5km, 0),
            # Nouvelles features debut de course
            'gradient_first_10km':  gradient_first_km(lats, lons, elevations, 10),
            'gradient_first_20km':  gradient_first_km(lats, lons, elevations, 20),
            'gradient_first_50km':  gradient_first_km(lats, lons, elevations, 50),
            'denivele_first_10km':  denivele_first_km(lats, lons, elevations, 10),
            'denivele_first_20km':  denivele_first_km(lats, lons, elevations, 20),
            'denivele_first_50km':  denivele_first_km(lats, lons, elevations, 50),
        }
    except Exception as e:
        print(f'  Erreur {os.path.basename(filepath)} : {e}')
        return {}

print('Fonctions chargees.')

Fonctions chargees.


In [6]:
# ── 5. Extraction GPX ─────────────────────────────────────────────────────────
gpx_features = []
for fname in tqdm(df['fichier_gpx']):
    if pd.isna(fname):
        gpx_features.append({})
        continue
    path = find_gpx_path(str(fname), GPX_DIR)
    if path:
        gpx_features.append(extract_gpx_features(path))
    else:
        gpx_features.append({})

df_gpx = pd.DataFrame(gpx_features)

# Pour les colonnes GPX : remplacer NaN par 0 si le fichier GPX existe
mask_avec_gpx = df_gpx['distance_gpx_km'].notna()
cols_gpx = list(df_gpx.columns)
df_gpx.loc[mask_avec_gpx, cols_gpx] = df_gpx.loc[mask_avec_gpx, cols_gpx].fillna(0)

print(f'Features GPX extraites : {df_gpx.shape}')
print(f'Avec GPX : {mask_avec_gpx.sum()}')
print(f'Sans GPX : {(~mask_avec_gpx).sum()}')

  0%|          | 0/7391 [00:00<?, ?it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/2864964498.py:101: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last_5km = float(np.sum(d for d in np.diff(elevations[-last_points:]) if d > 0))
  5%|▍         | 352/7391 [01:40<29:14,  4.01it/s]

  Erreur 2017 Koga Slag om Norg Rest day.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


 22%|██▏       | 1618/7391 [06:51<16:22,  5.87it/s]

  Erreur 2018 Tour of Fuzhou Stage 4.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


 22%|██▏       | 1625/7391 [06:52<10:58,  8.75it/s]

...

  Erreur 2024 Kreiz Breizh Elites Stage 2.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


 87%|████████▋ | 6424/7391 [31:05<04:59,  3.23it/s]

  Erreur 2024 Tre Valli Varesine.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


100%|██████████| 7391/7391 [34:31<00:00,  3.57it/s]

Features GPX extraites : (7391, 25)
Avec GPX : 7358
Sans GPX : 33


In [7]:
# ── 6. Assemblage final ───────────────────────────────────────────────────────
cols_matching = ['jour', 'mois', 'annee', 'stage', 'nom_pcs', 'categorie', 'fichier_gpx']
cols_komoot_all = ['route_url'] + cols_komoot
cols_gpx_all = list(df_gpx.columns)

df_final = pd.concat([df.reset_index(drop=True), df_gpx.reset_index(drop=True)], axis=1)
cols_finales = [c for c in cols_matching + cols_komoot_all + cols_gpx_all if c in df_final.columns]
df_final = df_final[cols_finales]

print(f'Shape final : {df_final.shape}')
print(f'Colonnes : {list(df_final.columns)}')

Shape final : (7391, 50)
Colonnes : ['jour', 'mois', 'annee', 'stage', 'nom_pcs', 'categorie', 'fichier_gpx', 'route_url', 'road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unknown_km', 'path_km', 'unpaved_km', 'singletrack_km', 'compacted_gravel_km', 'alpine_km', 'ferry_km', 'alpine_path_km', 'distance_gpx_km', 'denivele_pos', 'denivele_neg', 'altitude_max', 'altitude_min', 'n_cols_cat4', 'n_cols_cat3', 'n_cols_cat2', 'n_cols_cat1', 'n_cols_hc', 'loc_last_col_cat4', 'loc_last_col_cat3', 'loc_last_col_cat2', 'loc_last_col_cat1', 'loc_last_col_hc', 'gradient_last_1km', 'gradient_last_3km', 'gradient_last_5km', 'denivele_last_5km', 'gradient_first_10km', 'gradient_first_20km', 'gradient_first_50km', 'denivele_first_10km', 'denivele_first_20km', 'denivele_first_50km']


In [8]:
# ── 7. Sauvegarde ─────────────────────────────────────────────────────────────
df_final.to_csv(OUTPUT_CSV, index=False)
print(f'Sauvegarde : {OUTPUT_CSV}')
print(f'Shape : {df_final.shape}')

print(f'\nCourses avec Komoot  : {df_final["route_url"].notna().sum()}')
print(f'Courses avec GPX     : {df_final["distance_gpx_km"].notna().sum()}')
print(f'Courses sans Komoot  : {df_final["route_url"].isna().sum()}')
print(f'Courses sans GPX     : {df_final["distance_gpx_km"].isna().sum()}')

print(f'\nNaN restants par colonne :')
nan_cols = df_final.isna().sum()
print(nan_cols[nan_cols > 0].sort_values(ascending=False))

Sauvegarde : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/dataset_final_v2.csv
Shape : (7391, 50)

Courses avec Komoot  : 7217
Courses avec GPX     : 7358
Courses sans Komoot  : 174
Courses sans GPX     : 33

NaN restants par colonne :
route_url                174
unknown_km               174
road_km                  174
alpine_path_km           174
ferry_km                 174
compacted_gravel_km      174
singletrack_km           174
unpaved_km               174
path_km                  174
alpine_km                174
cobblestones_km          174
asphalt_km               174
access_road_km           174
off-grid_(unknown)_km    174
cycleway_km              174
state_road_km            174
paved_km                 174
street_km                174
denivele_last_5km         33
loc_last_col_hc           33
gradient_last_1km         33
gradient_last_3km         33
gradient_last_5km         33
gradient_first_50km       33
gradient_first_10km       33
gradient_first_20km   

In [9]:
sans_gpx = df_final[df_final['distance_gpx_km'].isna()][['fichier_gpx', 'annee', 'categorie', 'nom_pcs']]
pd.set_option('display.max_rows', None)
print(sans_gpx.to_string(index=False))

                                                            fichier_gpx  annee categorie                               nom_pcs
                                    2017 Koga Slag om Norg Rest day.gpx   2017       1.1                          slag-om-norg
2018 Adriatica Ionica Race/Following the Serenissima Routes Stage 1.gpx   2018       2.1                 adriatica-ionica-race
2018 Adriatica Ionica Race/Following the Serenissima Routes Stage 2.gpx   2018       2.1                 adriatica-ionica-race
2018 Adriatica Ionica Race/Following the Serenissima Routes Stage 3.gpx   2018       2.1                 adriatica-ionica-race
2018 Adriatica Ionica Race/Following the Serenissima Routes Stage 4.gpx   2018       2.1                 adriatica-ionica-race
2018 Adriatica Ionica Race/Following the Serenissima Routes Stage 5.gpx   2018       2.1                 adriatica-ionica-race
                                        2018 Tour of Fuzhou Stage 4.gpx   2018       2.1                       

In [11]:
import pandas as pd

RACE_PATH = '/Users/arthurdeletang/Desktop/Stage M1/Code/race_data/race_features.csv'

df_race = pd.read_csv(RACE_PATH)

cols_gpx = ['distance_gpx_km', 'denivele_pos', 'altitude_max', 'n_cols_hc', 'gradient_last_1km']
cols_gpx = [c for c in cols_gpx if c in df_race.columns]

mask_zeros = (df_race[cols_gpx] == 0).all(axis=1)
print(f'Courses avec GPX = 0 partout : {mask_zeros.sum()}')
df_race[mask_zeros][['course', 'year', 'stage_num'] + cols_gpx].sort_values(['course', 'year'])

Courses avec GPX = 0 partout : 161


,course,year,stage_num,distance_gpx_km,denivele_pos,altitude_max,n_cols_hc,gradient_last_1km
220,4-jours-de-dunkerque,2017,1.0,0.0,0.0,0.0,0.0,0.0
223,4-jours-de-dunkerque,2017,2.0,0.0,0.0,0.0,0.0,0.0
225,4-jours-de-dunkerque,2017,3.0,0.0,0.0,0.0,0.0,0.0
227,4-jours-de-dunkerque,2017,4.0,0.0,0.0,0.0,0.0,0.0
230,4-jours-de-dunkerque,2017,5.0,0.0,0.0,0.0,0.0,0.0
234,4-jours-de-dunkerque,2017,6.0,0.0,0.0,0.0,0.0,0.0
987,4-jours-de-dunkerque,2018,1.0,0.0,0.0,0.0,0.0,0.0
990,4-jours-de-dunkerque,2018,2.0,0.0,0.0,0.0,0.0,0.0
991,4-jours-de-dunkerque,2018,3.0,0.0,0.0,0.0,0.0,0.0
995,4-jours-de-dunkerque,2018,4.0,0.0,0.0,0.0,0.0,0.0


In [12]:
import os, re
import numpy as np
import pandas as pd
import gpxpy
from scipy.signal import find_peaks
from tqdm import tqdm

GPX_DIR  = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
RACE_PATH = '/Users/arthurdeletang/Desktop/Stage M1/Code/race_data/race_features.csv'

df_race = pd.read_csv(RACE_PATH)

# Identifier les lignes avec GPX = 0 partout
cols_gpx_check = ['distance_gpx_km', 'denivele_pos', 'altitude_max']
cols_gpx_check = [c for c in cols_gpx_check if c in df_race.columns]
mask_zeros = (df_race[cols_gpx_check] == 0).all(axis=1)
print(f'Lignes avec GPX = 0 : {mask_zeros.sum()}')

# Voir quels fichier_gpx correspondent
# On reconstruit depuis nom_pcs + year + stage_num
# Chercher dans GPX_DIR les fichiers correspondants
def find_gpx_path(fname, gpx_dir):
    if pd.isna(fname):
        return None
    full = os.path.join(gpx_dir, fname)
    if os.path.exists(full):
        return full
    replaced = fname.replace(' / ', ' - ').replace('/', ' - ')
    if os.path.exists(os.path.join(gpx_dir, replaced)):
        return os.path.join(gpx_dir, replaced)
    base = fname.split(' / ')[0].split('/')[0].strip()
    for candidate in [base + ' .gpx', base + '.gpx']:
        if os.path.exists(os.path.join(gpx_dir, candidate)):
            return os.path.join(gpx_dir, candidate)
    return None

# Verifier que le fichier gpx existe pour les lignes à 0
if 'fichier_gpx' in df_race.columns:
    df_zeros = df_race[mask_zeros][['fichier_gpx']].copy()
    df_zeros['path'] = df_zeros['fichier_gpx'].apply(lambda x: find_gpx_path(str(x), GPX_DIR))
    print(f'Fichiers trouves : {df_zeros["path"].notna().sum()}/{len(df_zeros)}')
    print(df_zeros[df_zeros['path'].notna()].head(5))

Lignes avec GPX = 0 : 161


In [15]:
def parse_gpx_safe(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    content = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#)', '&amp;', content)
    return gpxpy.parse(content)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def get_number_peaks(series, prominence):
    store = find_peaks(series, prominence=prominence)
    try:
        if (len(store[0]) > 1) and (np.min(np.diff(store[0])) < prominence):
            return len(store[1]['prominences'][np.diff(np.insert(store[0], 0, 0)) > prominence])
        return len(store[1]['prominences'])
    except:
        return len(store[1]['prominences'])

def gradient_last_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1, 0, -1):
        dist_cumul += haversine(lats[i-1], lons[i-1], lats[i], lons[i])
        if dist_cumul >= km:
            return round((elevations[-1] - elevations[i-1]) / (km * 1000) * 100, 2)
    total = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    return round((elevations[-1] - elevations[0]) / (total * 1000) * 100, 2) if total > 0 else 0.0

def gradient_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        if dist_cumul >= km:
            return round((elevations[i+1] - elevations[0]) / (km * 1000) * 100, 2)
    total = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    return round((elevations[-1] - elevations[0]) / (total * 1000) * 100, 2) if total > 0 else 0.0

def denivele_first_km(lats, lons, elevations, km):
    dist_cumul, deniv = 0.0, 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        diff = elevations[i+1] - elevations[i]
        if diff > 0:
            deniv += diff
        if dist_cumul >= km:
            break
    return round(deniv, 0)

def extract_gpx_features(filepath):
    try:
        gpx = parse_gpx_safe(filepath)
        points = gpx.tracks[0].segments[0].points
        elevations = [p.elevation or 0 for p in points]
        lats = [p.latitude for p in points]
        lons = [p.longitude for p in points]
        if len(lats) < 2:
            return None

        dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
        elev_diff = np.diff(elevations)
        elev_series = elevations + [0]
        n = len(elev_series)

        def loc_last(p):
            peaks = find_peaks(elev_series, prominence=p)[0]
            return float(np.max(peaks) / n) if len(peaks) > 0 else 0.0

        last_pts = min(500, len(elevations))
        deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))

        return {
            'distance_gpx_km':   round(dist, 2),
            'denivele_pos':      round(float(np.sum(elev_diff[elev_diff > 0])), 0),
            'denivele_neg':      round(float(np.abs(np.sum(elev_diff[elev_diff < 0]))), 0),
            'altitude_max':      round(float(np.max(elevations)), 0),
            'altitude_min':      round(float(np.min(elevations)), 0),
            'n_cols_hc':  get_number_peaks(elev_series, 800),
            'n_cols_cat1': get_number_peaks(elev_series, 640) - get_number_peaks(elev_series, 800),
            'n_cols_cat2': get_number_peaks(elev_series, 320) - get_number_peaks(elev_series, 640),
            'n_cols_cat3': get_number_peaks(elev_series, 160) - get_number_peaks(elev_series, 320),
            'n_cols_cat4': get_number_peaks(elev_series, 80)  - get_number_peaks(elev_series, 160),
            'loc_last_col_cat4': round(loc_last(80), 4),
            'loc_last_col_cat3': round(loc_last(160), 4),
            'loc_last_col_cat2': round(loc_last(320), 4),
            'loc_last_col_cat1': round(loc_last(640), 4),
            'loc_last_col_hc':   round(loc_last(800), 4),
            'gradient_last_1km': gradient_last_km(lats, lons, elevations, 1),
            'gradient_last_3km': gradient_last_km(lats, lons, elevations, 3),
            'gradient_last_5km': gradient_last_km(lats, lons, elevations, 5),
            'denivele_last_5km': round(deniv_last, 0),
            'gradient_first_50km': gradient_first_km(lats, lons, elevations, 50),
            'denivele_first_50km': denivele_first_km(lats, lons, elevations, 50),
        }
    except Exception as e:
        print(f'  Erreur {os.path.basename(filepath)} : {e}')
        return None

# Corriger les lignes à 0
n_updated = 0
for idx in tqdm(df_race[mask_zeros].index):
    fname = df_race.loc[idx, 'fichier_gpx'] if 'fichier_gpx' in df_race.columns else None
    if pd.isna(fname):
        continue
    path = find_gpx_path(str(fname), GPX_DIR)
    if path is None:
        continue
    feats = extract_gpx_features(path)
    if feats is None:
        continue
    for col, val in feats.items():
        if col in df_race.columns:
            df_race.loc[idx, col] = val
    n_updated += 1

print(f'\nLignes mises a jour : {n_updated}')

# Verifier
mask_zeros_after = (df_race[cols_gpx_check] == 0).all(axis=1)
print(f'Lignes encore a 0 : {mask_zeros_after.sum()}')

# Sauvegarder
df_race.to_csv(RACE_PATH, index=False)
print(f'race_features.csv mis a jour')

100%|██████████| 161/161 [00:00<00:00, 549010.52it/s]


Lignes mises a jour : 0
Lignes encore a 0 : 161
race_features.csv mis a jour


In [16]:
print(df_race.columns.tolist())
print(df_race.head(2))

['date', 'year', 'course', 'stage', 'stage_num', 'classification', 'distance_gpx_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unpaved_km', 'compacted_gravel_km', 'road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'unknown_km', 'path_km', 'singletrack_km', 'denivele_pos', 'denivele_neg', 'altitude_max', 'altitude_min', 'n_cols_cat4', 'n_cols_cat3', 'n_cols_cat2', 'n_cols_cat1', 'n_cols_hc', 'loc_last_col_cat4', 'loc_last_col_cat3', 'loc_last_col_cat2', 'loc_last_col_cat1', 'loc_last_col_hc', 'gradient_last_1km', 'gradient_last_3km', 'gradient_last_5km', 'denivele_last_5km', 'gradient_first_50km', 'denivele_first_50km']
         date  year              course   stage  stage_num classification  \
0  2017-01-05  2017    nc-australia-itt  result        NaN             NC   
1  2017-01-06  2017  nc-new-zealand-itt  result        NaN             NC   

   distance_gpx_km  asphalt_km  paved_km  cobblestones_km  ...  \
0            40.79    

In [22]:
import os, re
import numpy as np
import pandas as pd
import gpxpy
from scipy.signal import find_peaks
from tqdm import tqdm

GPX_DIR = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
MATCH_DIR = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all'

# Charger tous les matchings pour avoir course -> fichier_gpx
dfs_match = []
for year in range(2017, 2026):
    path = os.path.join(MATCH_DIR, f'matching_{year}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df[df['statut'] == 'match']
        dfs_match.append(df[['nom_pcs', 'annee', 'stage', 'fichier_gpx']])

df_match_all = pd.concat(dfs_match, ignore_index=True)

# Creer un mapping course+year+stage -> fichier_gpx
def stage_to_num(s):
    if pd.isna(s) or s == 'result':
        return np.nan
    if 'stage_' in str(s):
        try:
            return float(str(s).replace('stage_', ''))
        except:
            return np.nan
    if s == 'prologue':
        return 0.0
    return np.nan

df_match_all['stage_num'] = df_match_all['stage'].apply(stage_to_num)

# Joindre avec race_features pour recuperer fichier_gpx
df_race_with_gpx = df_race.merge(
    df_match_all[['nom_pcs', 'annee', 'stage_num', 'fichier_gpx']].rename(
        columns={'nom_pcs': 'course', 'annee': 'year'}),
    on=['course', 'year', 'stage_num'],
    how='left'
)

# Dedupliquer en gardant la premiere correspondance
df_race_with_gpx = df_race_with_gpx.drop_duplicates(subset=['course', 'year', 'stage_num'])
print(f'Shape apres dedup : {df_race_with_gpx.shape}')

cols_check = ['distance_gpx_km', 'denivele_pos', 'altitude_max']
mask_zeros = (df_race_with_gpx[cols_check] == 0).all(axis=1)
print(f'Lignes a corriger : {mask_zeros.sum()}')

df_zeros = df_race_with_gpx[mask_zeros]
print(f'Avec fichier_gpx : {df_zeros["fichier_gpx"].notna().sum()}')
print(df_zeros[['course', 'year', 'stage_num', 'fichier_gpx']].head(10))

Shape apres dedup : (7391, 42)
Lignes a corriger : 161
Avec fichier_gpx : 160
                    course  year  stage_num  \
134  dwars-door-vlaanderen  2017        NaN   
176           gp-de-denain  2017        NaN   
220   4-jours-de-dunkerque  2017        1.0   
223   4-jours-de-dunkerque  2017        2.0   
225   4-jours-de-dunkerque  2017        3.0   
227   4-jours-de-dunkerque  2017        4.0   
230   4-jours-de-dunkerque  2017        5.0   
234   4-jours-de-dunkerque  2017        6.0   
260         gp-de-la-somme  2017        NaN   
353           slag-om-norg  2017        NaN   

                                           fichier_gpx  
134  2017 Dwars Door Vlaanderen - A travers la Flan...  
176  2017 GP de Denain - Porte du Hainaut - Valenci...  
220  2017 4 Jours de Dunkerque - Grand Prix des Hau...  
223  2017 4 Jours de Dunkerque - Grand Prix des Hau...  
225  2017 4 Jours de Dunkerque - Grand Prix des Hau...  
227  2017 4 Jours de Dunkerque - Grand Prix des Hau...  
230  

In [ ]:
n_updated = 0

for idx in tqdm(df_race_with_gpx[mask_zeros].index):
    fname = df_race_with_gpx.loc[idx, 'fichier_gpx']
    if pd.isna(fname):
        continue

    path = find_gpx_path(str(fname), GPX_DIR)
    if path is None:
        continue

    feats = extract_gpx_features(path)
    if feats is None:
        continue

    for col, val in feats.items():
        if col in df_race_with_gpx.columns:
            df_race_with_gpx.loc[idx, col] = val
    n_updated += 1

print(f'Lignes mises a jour : {n_updated}')

# Verifier
mask_zeros_after = (df_race_with_gpx[cols_check] == 0).all(axis=1)
print(f'Lignes encore a 0 : {mask_zeros_after.sum()}')

# Sauvegarder sans la colonne fichier_gpx
df_race_with_gpx.drop(columns=['fichier_gpx']).to_csv(RACE_PATH, index=False)
print('race_features.csv mis a jour')

  0%|          | 0/161 [00:00<?, ?it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
  1%|          | 1/161 [00:00<01:15,  2.11it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
  1%|          | 2/161 [00:00<01:05,  2.43it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a differen

  Erreur 2017 Koga Slag om Norg Rest day.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
  7%|▋         | 12/161 [00:03<00:32,  4.56it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
  8%|▊         | 13/161 [00:03<00:37,  3.96it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(gene

  Erreur 2018 Tour of Fuzhou Stage 4.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)
  Erreur 2018 Africa Cup - TTT.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)
  Erreur 2018 Africa Cup - ITT.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)
  Erreur 2018 Africa Cup.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
 24%|██▎       | 38/161 [00:08<00:15,  8.06it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
 24%|██▍       | 39/161 [00:09<00:19,  6.27it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(gene

...

  Erreur 2024 Kreiz Breizh Elites Stage 2.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
 84%|████████▍ | 136/161 [00:35<00:04,  6.06it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
 85%|████████▌ | 137/161 [00:35<00:03,  6.25it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(ge

  Erreur 2024 Tre Valli Varesine.gpx : Error parsing XML: Specification mandates value for attribute async, line 161, column 15 (<string>, line 161)


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
 89%|████████▉ | 143/161 [00:36<00:03,  4.53it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last = float(np.sum(d for d in np.diff(elevations[-last_pts:]) if d > 0))
 89%|████████▉ | 144/161 [00:37<00:03,  4.68it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_67721/866117685.py:71: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(ge

Lignes mises a jour : 142
Lignes encore a 0 : 19
race_features.csv mis a jour


In [24]:
df_race = pd.read_csv(RACE_PATH)
mask = df_race['distance_gpx_km'] == 0
print(f'Lignes avec distance_gpx_km = 0 : {mask.sum()}')
df_race[mask][['course', 'year', 'stage_num', 'distance_gpx_km']].sort_values(['course', 'year'])

Lignes avec distance_gpx_km = 0 : 19


,course,year,stage_num,distance_gpx_km
1623,africa-cup,2018,NaN,0.0
1622,africa-cup-itt,2018,NaN,0.0
1621,africa-cup-ttt,2018,NaN,0.0
4015,african-continental-championships-itt,2022,NaN,0.0
4009,african-cycling-championships-ttt,2022,NaN,0.0
5432,grand-prix-el-marsa,2023,NaN,0.0
6209,kreiz-breizh-elites,2024,2.0,0.0
2469,nc-china,2019,NaN,0.0
3030,nc-china,2020,NaN,0.0
2258,nc-dominican-republic,2019,NaN,0.0


In [25]:
df_race = pd.read_csv(RACE_PATH)
mask = df_race['distance_gpx_km'] == 0
print(f'Lignes supprimees : {mask.sum()}')
df_race = df_race[~mask]
print(f'Shape apres suppression : {df_race.shape}')
df_race.to_csv(RACE_PATH, index=False)
print('race_features.csv mis a jour')

Lignes supprimees : 19
Shape apres suppression : (7372, 41)
race_features.csv mis a jour
